# Periodogram

This tutorial demonstrates the Lomb–Scargle periodogram via `light_curve.Periodogram`,
including parameter choices, power spectrum extraction, and feature extraction on the
spectrum and phase-folded light curve.

## Synthetic periodic light curve

In [ ]:
import light_curve as licu
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
PERIOD = 23.5  # days
t = np.sort(rng.uniform(0, 300, 150))
m = 15.0 + 0.4 * np.sin(2 * np.pi * t / PERIOD) + rng.normal(0, 0.04, 150)
err = np.full(150, 0.04)
print(f'True period: {PERIOD} days, n_obs: {len(t)}')

## Basic usage

`Periodogram(peaks=N)` returns the `N` strongest periods and their signal-to-noise
ratios:

$$\text{S/N}(\omega_\text{peak}) = \frac{P(\omega_\text{peak}) - \langle P(\omega) \rangle}{\sigma_{P(\omega)}}$$

In [ ]:
pg = licu.Periodogram(peaks=1, fast=True)
result = pg(t, m, err)
for name, val in zip(pg.names, result):
    print(f'  {name}: {val:.4f}')

## Key parameters and recommendations

| Parameter | Default | What it controls | Recommendation |
|-----------|---------|-----------------|----------------|
| `peaks` | 1 | Number of period/S\N pairs returned | 1–5 for most surveys |
| `resolution` | 10 | Frequency grid oversampling factor | Increase to 20–50 for faint periods |
| `max_freq_factor` | 1.0 | Nyquist factor upper limit on frequency | Keep ≥ 1; raise to 2–3 for rapid variables |
| `nyquist` | `'average'` | How to estimate the Nyquist frequency | `'average'` works well; `'median'` for sparse gaps |
| `fast` | `True` | Use NFFT-based O(N log N) algorithm | `True` unless you need exact O(N²) values |
| `normalization` | `'psd'` | Power normalisation | `'psd'` for S/N comparisons; `'standard'` ∈ [0, 1] |

Higher `resolution` and `max_freq_factor` increase both accuracy and runtime.

## Multiple peaks

In [ ]:
pg5 = licu.Periodogram(peaks=5, resolution=20)
result5 = pg5(t, m, err)
print('Top 5 periods (days):')
for i in range(5):
    print(f'  peak {i+1}: period = {result5[i*2]:.3f} d, S/N = {result5[i*2+1]:.2f}')

## Extracting and plotting the power spectrum

`Periodogram` returns only the top-N peak periods and their S/N ratios.
To visualise the full power spectrum, compute it with `astropy.timeseries.LombScargle`
using the same frequency grid that `light_curve` uses internally:

In [ ]:
from astropy.timeseries import LombScargle

# Estimate the frequency grid as light_curve does internally:
# min_freq = 1 / T_baseline,  max_freq = nyquist * max_freq_factor,
# with oversampling factor = resolution
T_baseline = t.max() - t.min()
dt_median  = np.median(np.diff(np.sort(t)))
freq_min = 1.0 / T_baseline
freq_max = 1.0 / (2 * dt_median)  # Nyquist
resolution = 10
n_freq = int(resolution * freq_max / freq_min)
freqs = np.linspace(freq_min, freq_max, n_freq)

ls = LombScargle(t, m, err)
powers = ls.power(freqs)

best_idx = np.argmax(powers)
best_period = 1.0 / freqs[best_idx]

fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(1.0 / freqs, powers, lw=0.7, color='steelblue')
ax.axvline(best_period, color='red', lw=1.5, ls='--', label=f'Best period = {best_period:.2f} d')
ax.axvline(PERIOD, color='green', lw=1.5, ls=':', label=f'True period = {PERIOD} d')
ax.set_xscale('log')
ax.set_xlabel('Period (days)')
ax.set_ylabel('LS power')
ax.set_title('Lomb–Scargle power spectrum (astropy)')
ax.legend()
plt.tight_layout()
plt.show()

# light_curve Periodogram gives the same best period
pg_best = licu.Periodogram(peaks=1, resolution=resolution)
lc_result = pg_best(t, m, err)
print(f'astropy best period : {best_period:.3f} d'
      f'   (S/N from light_curve: {lc_result[1]:.2f})')

## Feature extraction on the power spectrum

`Periodogram` accepts a `features` argument: a list of feature extractors that are
applied to the **power spectrum** (treated as an unevenly sampled time series over
frequencies). This allows extracting shape statistics from the spectrum itself:

In [ ]:
# Extract statistical features from the power spectrum
# features= accepts a list of feature extractors applied to the spectrum
pg_with_spectrum_features = licu.Periodogram(
    peaks=1,
    features=[
        licu.Amplitude(),
        licu.Kurtosis(),
        licu.Skew(),
        licu.BeyondNStd(nstd=2),
    ],
)
result_sf = pg_with_spectrum_features(t, m, err)
print('Periodogram + spectrum features:')
for name, val in zip(pg_with_spectrum_features.names, result_sf):
    print(f'  {name:50s} = {val:.4f}')

## Feature extraction on the phase-folded light curve

`phase_features` applies extractors to the light curve **phase-folded** at the best period
(phase runs 0 → 1, with phase 0 at magnitude minimum). This captures light curve shape
directly — useful for distinguishing eclipsing binaries from pulsating stars:

In [ ]:
pg_phase = licu.Periodogram(
    peaks=1,
    phase_features=[
        licu.Amplitude(),
        licu.Skew(),
        licu.StetsonK(),
    ],
)
result_pf = pg_phase(t, m, err)
print('Phase-folded features (prefixed with period_folded_):')
for name, val in zip(pg_phase.names, result_pf):
    print(f'  {name:50s} = {val:.4f}')

# Plot the phase-folded light curve
best_p = result_pf[0]
phase = (t % best_p) / best_p
fig, ax = plt.subplots(figsize=(6, 3))
ax.errorbar(phase, m, yerr=err, fmt='.', alpha=0.5)
ax.set_xlabel('Phase')
ax.set_ylabel('Magnitude')
ax.invert_yaxis()
ax.set_title(f'Phase-folded light curve (P = {best_p:.2f} d)')
plt.tight_layout()
plt.show()

## Notes

- `fast=True` (default) uses an NFFT-based algorithm; `fast=False` uses the exact O(N²) implementation.
- Periods are in the same units as the input time array.
- S/N is computed as `(peak_power − mean_power) / std_power` over the frequency grid.
- See [API reference](api/periodogram.md) for all constructor options.